# Run 3 — YOLO11m + Copy-Paste Augmentation

**Epic:** TTV-118 | **Config:** `yolo11m_copypaste.yaml`

Aplica copy-paste offline 3x sobre `competidor_number` (clase 3) antes de entrenar.

**Hipótesis:** Oversampling offline de clase minoritaria mejora AP sin dañar otras clases

**Basado en:** Kisantal 2019 — reporta +7.1% AP small-object con copy-paste

---

## 0. Verificar GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "ERROR: No GPU detectada. Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Instalar dependencias

In [ ]:
!pip install -q roboflow ultralytics

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/cycling-photo-ai/experiments'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Output dir: {DRIVE_OUTPUT}")

## 3. Descargar dataset v1

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "xOdnFACkI2vaUzBKVRic"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("titan-ca4ce").project("titan-detection-jedpa")
version = project.version(7)

dataset = version.download("yolov11", location="/content/dataset_v1")
print("Dataset descargado")

## 4. Copy-Paste Augmentation para `competidor_number`

Recorta instancias de `competidor_number` (class_id=3) y las pega en imágenes de train
que no tienen esa clase. Variación: escala ±20%, rotación ±5°, brillo ±10%.

Objetivo: 3x oversampling.

In [ ]:
import random
import shutil
from pathlib import Path

import numpy as np
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Copiar dataset original para no modificarlo
src_dir = Path("/content/dataset_v1")
aug_dir = Path("/content/dataset_v1_copypaste")

if aug_dir.exists():
    shutil.rmtree(aug_dir)
shutil.copytree(src_dir, aug_dir)

print(f"Dataset copiado a {aug_dir}")

In [ ]:
TARGET_CLASS_ID = 3  # competidor_number
MULTIPLIER = 3
SCALE_RANGE = (0.8, 1.2)
ROTATION_RANGE = (-5.0, 5.0)
BRIGHTNESS_RANGE = (0.9, 1.1)

images_dir = aug_dir / "train" / "images"
labels_dir = aug_dir / "train" / "labels"

def find_image(stem):
    for ext in [".jpg", ".jpeg", ".png"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

# 1. Separar imágenes con y sin target class
source_pairs = []  # (img_path, [label_lines_with_target])
target_images = []  # imgs sin la clase target

for label_file in sorted(labels_dir.glob("*.txt")):
    lines = [l for l in label_file.read_text().strip().split("\n") if l.strip()]
    has_target = any(l.split()[0] == str(TARGET_CLASS_ID) for l in lines)
    
    img_path = find_image(label_file.stem)
    if img_path is None:
        continue
    
    if has_target:
        target_lines = [l for l in lines if l.split()[0] == str(TARGET_CLASS_ID)]
        source_pairs.append((img_path, label_file, target_lines))
    else:
        target_images.append((img_path, label_file))

print(f"Imágenes con competidor_number: {len(source_pairs)}")
print(f"Imágenes sin competidor_number (destinos): {len(target_images)}")

In [ ]:
# 2. Ejecutar copy-paste
count = 0
errors = 0

for mult in range(MULTIPLIER):
    for src_img_path, src_label_path, src_target_lines in source_pairs:
        if not target_images:
            break
        
        dst_img_path, dst_label_path = random.choice(target_images)
        
        try:
            src_img = Image.open(src_img_path).convert("RGB")
            dst_img = Image.open(dst_img_path).convert("RGB").copy()
            
            new_labels = []
            
            for label_line in src_target_lines:
                parts = label_line.split()
                cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                
                # Crop from source
                sw, sh = src_img.size
                x1 = max(0, int((cx - w/2) * sw))
                y1 = max(0, int((cy - h/2) * sh))
                x2 = min(sw, int((cx + w/2) * sw))
                y2 = min(sh, int((cy + h/2) * sh))
                
                if x2 <= x1 or y2 <= y1:
                    continue
                    
                crop = src_img.crop((x1, y1, x2, y2))
                
                # Random transforms
                scale = random.uniform(*SCALE_RANGE)
                new_w = max(1, int(crop.width * scale))
                new_h = max(1, int(crop.height * scale))
                crop = crop.resize((new_w, new_h), Image.LANCZOS)
                
                angle = random.uniform(*ROTATION_RANGE)
                crop = crop.rotate(angle, expand=True, fillcolor=(0, 0, 0))
                
                brightness = random.uniform(*BRIGHTNESS_RANGE)
                crop_arr = np.array(crop, dtype=np.float32) * brightness
                crop = Image.fromarray(np.clip(crop_arr, 0, 255).astype(np.uint8))
                
                # Paste at random position
                dw, dh = dst_img.size
                max_x = dw - crop.width
                max_y = dh - crop.height
                if max_x <= 0 or max_y <= 0:
                    continue
                
                paste_x = random.randint(0, max_x)
                paste_y = random.randint(0, max_y)
                dst_img.paste(crop, (paste_x, paste_y))
                
                # New YOLO label
                new_cx = (paste_x + crop.width / 2) / dw
                new_cy = (paste_y + crop.height / 2) / dh
                new_w_norm = crop.width / dw
                new_h_norm = crop.height / dh
                new_labels.append(f"{TARGET_CLASS_ID} {new_cx:.6f} {new_cy:.6f} {new_w_norm:.6f} {new_h_norm:.6f}")
            
            if not new_labels:
                continue
            
            # Save augmented image + labels with unique name
            aug_name = f"{dst_img_path.stem}_cp{count}"
            dst_img.save(images_dir / f"{aug_name}{dst_img_path.suffix}")
            
            # Copy original labels + add new ones
            orig_labels = dst_label_path.read_text().strip()
            all_labels = orig_labels + "\n" + "\n".join(new_labels)
            (labels_dir / f"{aug_name}.txt").write_text(all_labels)
            
            count += 1
            
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"Error: {e}")

print(f"\nCopy-paste completado: {count} imágenes augmentadas creadas ({errors} errores)")

In [ ]:
# 3. Verificar resultado
orig_train_imgs = len(list((src_dir / "train" / "images").glob("*.*")))
aug_train_imgs = len(list((aug_dir / "train" / "images").glob("*.*")))

# Contar competidor_number antes y después
def count_class(labels_dir, class_id):
    total = 0
    for f in labels_dir.glob("*.txt"):
        for line in f.read_text().strip().split("\n"):
            if line.strip() and line.split()[0] == str(class_id):
                total += 1
    return total

orig_count = count_class(src_dir / "train" / "labels", TARGET_CLASS_ID)
aug_count = count_class(aug_dir / "train" / "labels", TARGET_CLASS_ID)

print(f"Train images: {orig_train_imgs} → {aug_train_imgs} (+{aug_train_imgs - orig_train_imgs})")
print(f"competidor_number annotations: {orig_count} → {aug_count} ({aug_count/orig_count:.1f}x)")

## 5. Actualizar data.yaml

In [ ]:
# data.yaml apunta a rutas relativas, funciona igual
data_yaml = aug_dir / "data.yaml"
print(data_yaml.read_text())

## 6. Reproducibilidad

In [ ]:
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print(f"Seed: {SEED}")

## 7. Entrenar YOLO11m — Run 3 Copy-Paste

Misma config que Run 2 (mixup=0.1, cls_pw=0.5) pero sobre dataset augmentado.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "run3_yolo11m_copypaste"

model = YOLO("yolo11m.pt")

results = model.train(
    data=str(aug_dir / "data.yaml"),
    epochs=200,
    patience=30,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=7.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.1,
    cutmix=0.1,
    cls_pw=0.5,
    save_json=True,
    deterministic=True,
    seed=SEED,
    project="/content/experiments",
    name=RUN_NAME,
)

## 8. Revisar resultados

In [ ]:
import pandas as pd

run_dir = Path(f"/content/experiments/{RUN_NAME}")

print("=== Métricas finales ===")
for key, val in results.results_dict.items():
    print(f"  {key}: {val:.4f}")

results_csv = run_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"\nEpochs entrenados: {len(df)}")
    print(f"Mejor mAP@0.5: {df['metrics/mAP50(B)'].max():.4f} (epoch {df['metrics/mAP50(B)'].idxmax()})")
    print(f"Mejor mAP@0.5:0.95: {df['metrics/mAP50-95(B)'].max():.4f} (epoch {df['metrics/mAP50-95(B)'].idxmax()})")

In [ ]:
import matplotlib.pyplot as plt

if results_csv.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(df['train/box_loss'], label='train box')
    axes[0].plot(df['train/cls_loss'], label='train cls')
    axes[0].plot(df['val/box_loss'], label='val box', linestyle='--')
    axes[0].plot(df['val/cls_loss'], label='val cls', linestyle='--')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].set_xlabel('Epoch')

    axes[1].plot(df['metrics/mAP50(B)'], label='mAP@0.5')
    axes[1].plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axes[1].axhline(y=0.80, color='r', linestyle=':', label='Target 0.80')
    axes[1].set_title('mAP')
    axes[1].legend()
    axes[1].set_xlabel('Epoch')

    axes[2].plot(df['metrics/precision(B)'], label='Precision')
    axes[2].plot(df['metrics/recall(B)'], label='Recall')
    axes[2].set_title('Precision / Recall')
    axes[2].legend()
    axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(run_dir / 'training_curves.png', dpi=150)
    plt.show()

## 9. Validación per-class

In [ ]:
# Evaluar sobre DATASET ORIGINAL (no augmentado) para comparación justa
best_model = YOLO(str(run_dir / "weights" / "best.pt"))
val_results = best_model.val(data="/content/dataset_v1/data.yaml", imgsz=640, save_json=True)

class_names = ['bicycle', 'bicycle_text', 'clothes_text', 'competidor_number', 'cyclist',
               'cyclist_clothes', 'cyclist_with_bike', 'helmet', 'helmet_text', 'objects']

print("\n=== Per-class AP@0.5 ===")
for i, name in enumerate(class_names):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    ap = val_results.box.ap[i] if i < len(val_results.box.ap) else 0
    print(f"  {name:25s} AP@0.5={ap50:.4f}  AP@0.5:0.95={ap:.4f}")

## 10. Confusion matrix + PR curves

In [ ]:
from IPython.display import Image, display

for fname, title in [("confusion_matrix_normalized.png", "Confusion Matrix"),
                      ("PR_curve.png", "PR Curves"),
                      ("val_batch0_pred.jpg", "Sample predictions")]:
    fpath = run_dir / fname
    if fpath.exists():
        print(f"\n{title}:")
        display(Image(filename=str(fpath), width=800))

## 11. Guardar en Google Drive

In [ ]:
drive_run_dir = Path(DRIVE_OUTPUT) / RUN_NAME
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(run_dir, drive_run_dir)

weights_size = (drive_run_dir / "weights" / "best.pt").stat().st_size / 1e6
print(f"Guardado en: {drive_run_dir}")
print(f"best.pt: {weights_size:.1f} MB")

## 12. Resumen para EXPERIMENT_LOG.md

In [ ]:
print("="*60)
print("RESUMEN PARA EXPERIMENT_LOG.md")
print("="*60)
print(f"\n### Run 3 — YOLO11m + Copy-Paste")
print(f"- **Fecha:** {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print(f"- **Config:** yolo11m_copypaste.yaml")
print(f"- **Dataset:** v1 + copy-paste 3x competidor_number")
print(f"- **GPU:** {torch.cuda.get_device_name(0)}")
print(f"- **Augmented images added:** {aug_train_imgs - orig_train_imgs}")
print(f"- **competidor_number annotations:** {orig_count} → {aug_count} ({aug_count/orig_count:.1f}x)")
print(f"- **Epochs entrenados:** {len(df)}")
print(f"- **Mejor epoch:** {df['metrics/mAP50(B)'].idxmax()}")
print(f"")
print(f"| Métrica | Valor |")
print(f"|---|---|")
print(f"| mAP@0.5 | {df['metrics/mAP50(B)'].max():.4f} |")
print(f"| mAP@0.5:0.95 | {df['metrics/mAP50-95(B)'].max():.4f} |")
print(f"| Precision | {df['metrics/precision(B)'].max():.4f} |")
print(f"| Recall | {df['metrics/recall(B)'].max():.4f} |")
print(f"")
print(f"**Per-class AP@0.5:**")
print(f"")
print(f"| Clase | AP@0.5 |")
print(f"|---|---|")
for i, name in enumerate(class_names):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    print(f"| {name} | {ap50:.4f} |")
print(f"\nPesos guardados en: {drive_run_dir}/weights/best.pt")